<a href="https://colab.research.google.com/github/Renyambar/Reny_MachineLearning/blob/main/JS09/JS09_Tugas1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Tugas Lab 1 — Benchmark Approximate Nearest Neighbors (ANN)

In [1]:
!pip install -q annoy faiss-cpu hnswlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import numpy as np
import pandas as pd
import time
import matplotlib.pyplot as plt

from annoy import AnnoyIndex
import faiss
import hnswlib

from sklearn.neighbors import NearestNeighbors
from IPython.display import display

import warnings
warnings.filterwarnings("ignore")

print("Semua library berhasil diimpor.")

Semua library berhasil diimpor.


In [3]:
# Parameter eksperimen
K = 10
N_QUERY = 100

# Parameter ANNOY
ANNOY_TREES = 10

# Parameter FAISS
FAISS_NPROBE = 10

# Parameter HNSW
HNSW_M = 16
HNSW_EF_CONSTRUCTION = 100
HNSW_EF = 100

# Seed agar eksperimen dapat direproduksi
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)

print("Konfigurasi eksperimen:")
print("K             :", K)
print("Jumlah Query  :", N_QUERY)
print("ANNOY Trees   :", ANNOY_TREES)
print("HNSW M        :", HNSW_M)

Konfigurasi eksperimen:
K             : 10
Jumlah Query  : 100
ANNOY Trees   : 10
HNSW M        : 16


In [4]:
def generate_dataset(n_samples, dimensions, seed=42):
    rng = np.random.default_rng(seed)

    data = rng.random(
        (n_samples, dimensions),
        dtype=np.float32
    )

    return data


# Dataset 1: 1000 data, 2 dimensi
data_1000_2d = generate_dataset(
    n_samples=1000,
    dimensions=2
)

# Dataset 2: 1 juta data, 5 dimensi
data_1m_5d = generate_dataset(
    n_samples=1000000,
    dimensions=5
)

print("Dataset berhasil dibuat.")
print("Dataset 1000/2D :", data_1000_2d.shape)
print("Dataset 1M/5D   :", data_1m_5d.shape)

Dataset berhasil dibuat.
Dataset 1000/2D : (1000, 2)
Dataset 1M/5D   : (1000000, 5)


In [5]:
print("Informasi Dataset")
print("=" * 40)

print("Dataset 1:")
print("Jumlah data :", len(data_1000_2d))
print("Dimensi     :", data_1000_2d.shape[1])

print("\nDataset 2:")
print("Jumlah data :", len(data_1m_5d))
print("Dimensi     :", data_1m_5d.shape[1])

Informasi Dataset
Dataset 1:
Jumlah data : 1000
Dimensi     : 2

Dataset 2:
Jumlah data : 1000000
Dimensi     : 5


In [6]:
def get_exact_neighbors(data, query_ids, metric, k=10):

    model = NearestNeighbors(
        n_neighbors=k + 1,
        algorithm="brute",
        metric=metric,
        n_jobs=-1
    )

    model.fit(data)

    distances, indices = model.kneighbors(
        data[query_ids]
    )

    exact_results = []

    for i, query_id in enumerate(query_ids):

        # Menghilangkan data query itu sendiri
        neighbors = [
            idx for idx in indices[i]
            if idx != query_id
        ]

        exact_results.append(neighbors[:k])

    return exact_results

In [7]:
def calculate_recall(exact_results, approximate_results):

    recalls = []

    for exact, approximate in zip(
        exact_results,
        approximate_results
    ):

        exact_set = set(exact)
        approximate_set = set(approximate)

        intersection = exact_set.intersection(
            approximate_set
        )

        recall = len(intersection) / len(exact_set)

        recalls.append(recall)

    return np.mean(recalls)

In [12]:
def test_annoy(data, query_ids, metric, k=10):

    n_samples, dimensions = data.shape

    annoy_metric = (
        "euclidean" if metric == "euclidean"
        else "angular"
    )

    index = AnnoyIndex(dimensions, annoy_metric)

    # Proses pembangunan index
    start_build = time.perf_counter()

    for i, vector in enumerate(data):
        index.add_item(i, vector.tolist())

    index.build(ANNOY_TREES)

    build_time = time.perf_counter() - start_build

    # Proses pencarian
    approximate_results = []

    start_query = time.perf_counter()

    for query_id in query_ids:

        neighbors = index.get_nns_by_item(
            int(query_id),
            k + 1,
            search_k=-1
        )

        neighbors = [
            idx for idx in neighbors
            if idx != query_id
        ]

        approximate_results.append(neighbors[:k])

    query_time = (
        time.perf_counter() - start_query
    ) / len(query_ids)

    return approximate_results, build_time, query_time

In [13]:
def test_faiss(data, query_ids, metric, k=10):

    n_samples, dimensions = data.shape

    # Menyiapkan data berdasarkan metrik
    indexed_data = data.copy()

    if metric == "cosine":

        faiss.normalize_L2(indexed_data)

        metric_type = faiss.METRIC_INNER_PRODUCT

        quantizer = faiss.IndexFlatIP(dimensions)

    else:

        metric_type = faiss.METRIC_L2

        quantizer = faiss.IndexFlatL2(dimensions)

    nlist = min(100, max(1, int(np.sqrt(n_samples))))

    index = faiss.IndexIVFFlat(
        quantizer,
        dimensions,
        nlist,
        metric_type
    )

    # Membangun index
    start_build = time.perf_counter()

    index.train(indexed_data)
    index.add(indexed_data)

    index.nprobe = min(FAISS_NPROBE, nlist)

    build_time = time.perf_counter() - start_build

    # Menyiapkan query
    queries = data[query_ids].copy()

    if metric == "cosine":
        faiss.normalize_L2(queries)

    # Pencarian
    start_query = time.perf_counter()

    distances, indices = index.search(
        queries,
        k + 1
    )

    query_time = (
        time.perf_counter() - start_query
    ) / len(query_ids)

    approximate_results = []

    for i, query_id in enumerate(query_ids):

        neighbors = [
            int(idx) for idx in indices[i]
            if idx >= 0 and idx != query_id
        ]

        approximate_results.append(neighbors[:k])

    return approximate_results, build_time, query_time

In [14]:
def test_hnsw(data, query_ids, metric, k=10):

    n_samples, dimensions = data.shape

    space = (
        "l2" if metric == "euclidean"
        else "cosine"
    )

    index = hnswlib.Index(
        space=space,
        dim=dimensions
    )

    # Membangun index
    start_build = time.perf_counter()

    index.init_index(
        max_elements=n_samples,
        ef_construction=HNSW_EF_CONSTRUCTION,
        M=HNSW_M
    )

    ids = np.arange(n_samples)

    index.add_items(data, ids)

    index.set_ef(HNSW_EF)

    build_time = time.perf_counter() - start_build

    # Pencarian
    queries = data[query_ids]

    start_query = time.perf_counter()

    labels, distances = index.knn_query(
        queries,
        k=k + 1
    )

    query_time = (
        time.perf_counter() - start_query
    ) / len(query_ids)

    approximate_results = []

    for i, query_id in enumerate(query_ids):

        neighbors = [
            int(idx) for idx in labels[i]
            if idx != query_id
        ]

        approximate_results.append(neighbors[:k])

    return approximate_results, build_time, query_time

In [15]:
results = []

def run_benchmark(data, dataset_label, metric):

    n_samples = len(data)

    rng = np.random.default_rng(RANDOM_STATE)

    query_ids = rng.choice(
        n_samples,
        size=min(N_QUERY, n_samples),
        replace=False
    )

    print("\n" + "=" * 60)
    print(f"Dataset: {dataset_label}")
    print(f"Metrik : {metric}")
    print("=" * 60)

    # Exact Nearest Neighbor
    print("Menghitung Exact Nearest Neighbor...")

    exact_results = get_exact_neighbors(
        data,
        query_ids,
        metric,
        K
    )

    # ANNOY
    print("Menjalankan ANNOY...")

    annoy_results, annoy_build, annoy_query = test_annoy(
        data,
        query_ids,
        metric,
        K
    )

    annoy_recall = calculate_recall(
        exact_results,
        annoy_results
    )

    # FAISS
    print("Menjalankan FAISS...")

    faiss_results, faiss_build, faiss_query = test_faiss(
        data,
        query_ids,
        metric,
        K
    )

    faiss_recall = calculate_recall(
        exact_results,
        faiss_results
    )

    # HNSW
    print("Menjalankan HNSW...")

    hnsw_results, hnsw_build, hnsw_query = test_hnsw(
        data,
        query_ids,
        metric,
        K
    )

    hnsw_recall = calculate_recall(
        exact_results,
        hnsw_results
    )

    # Menyimpan hasil
    results.append({
        "Jumlah Data/Dimensi": dataset_label,
        "Metrik": metric,
        "ANNOY": f"{annoy_query * 1000:.3f} ms | Recall {annoy_recall:.3f}",
        "FAISS": f"{faiss_query * 1000:.3f} ms | Recall {faiss_recall:.3f}",
        "HNSW": f"{hnsw_query * 1000:.3f} ms | Recall {hnsw_recall:.3f}",
        "ANNOY Build (s)": annoy_build,
        "FAISS Build (s)": faiss_build,
        "HNSW Build (s)": hnsw_build,
        "ANNOY Query (ms)": annoy_query * 1000,
        "FAISS Query (ms)": faiss_query * 1000,
        "HNSW Query (ms)": hnsw_query * 1000,
        "ANNOY Recall": annoy_recall,
        "FAISS Recall": faiss_recall,
        "HNSW Recall": hnsw_recall
    })

    print("Benchmark selesai.")

In [16]:
run_benchmark(
    data_1000_2d,
    "1000/2D",
    "euclidean"
)


Dataset: 1000/2D
Metrik : euclidean
Menghitung Exact Nearest Neighbor...
Menjalankan ANNOY...
Menjalankan FAISS...
Menjalankan HNSW...
Benchmark selesai.


In [17]:
run_benchmark(
    data_1000_2d,
    "1000/2D",
    "cosine"
)


Dataset: 1000/2D
Metrik : cosine
Menghitung Exact Nearest Neighbor...
Menjalankan ANNOY...
Menjalankan FAISS...
Menjalankan HNSW...
Benchmark selesai.


In [18]:
run_benchmark(
    data_1m_5d,
    "1000000/5D",
    "euclidean"
)


Dataset: 1000000/5D
Metrik : euclidean
Menghitung Exact Nearest Neighbor...
Menjalankan ANNOY...
Menjalankan FAISS...
Menjalankan HNSW...
Benchmark selesai.


In [19]:
run_benchmark(
    data_1m_5d,
    "1000000/5D",
    "cosine"
)


Dataset: 1000000/5D
Metrik : cosine
Menghitung Exact Nearest Neighbor...
Menjalankan ANNOY...
Menjalankan FAISS...
Menjalankan HNSW...
Benchmark selesai.


In [20]:
df_results = pd.DataFrame(results)

final_table = df_results[
    [
        "Jumlah Data/Dimensi",
        "Metrik",
        "ANNOY",
        "FAISS",
        "HNSW"
    ]
]

display(final_table)

,Jumlah Data/Dimensi,Metrik,ANNOY,FAISS,HNSW
0,1000/2D,euclidean,0.020 ms | Recall 0.996,0.005 ms | Recall 1.000,0.011 ms | Recall 1.000
1,1000/2D,cosine,0.019 ms | Recall 0.999,0.004 ms | Recall 0.999,0.010 ms | Recall 0.999
2,1000000/5D,euclidean,0.054 ms | Recall 0.954,0.524 ms | Recall 1.000,0.053 ms | Recall 1.000
3,1000000/5D,cosine,0.054 ms | Recall 0.982,0.651 ms | Recall 1.000,0.050 ms | Recall 1.000


In [21]:
summary_table = df_results[
    [
        "Jumlah Data/Dimensi",
        "ANNOY",
        "FAISS",
        "HNSW"
    ]
].copy()

display(
    summary_table.style
    .set_properties(**{
        "text-align": "center",
        "padding": "12px"
    })
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "center"),
                ("background-color", "#e9edf5"),
                ("font-weight", "bold")
            ]
        }
    ])
)

,Jumlah Data/Dimensi,ANNOY,FAISS,HNSW
0,1000/2D,0.020 ms | Recall 0.996,0.005 ms | Recall 1.000,0.011 ms | Recall 1.000
1,1000/2D,0.019 ms | Recall 0.999,0.004 ms | Recall 0.999,0.010 ms | Recall 0.999
2,1000000/5D,0.054 ms | Recall 0.954,0.524 ms | Recall 1.000,0.053 ms | Recall 1.000
3,1000000/5D,0.054 ms | Recall 0.982,0.651 ms | Recall 1.000,0.050 ms | Recall 1.000
